# 04 · MLP with category embeddings: ShopSphere purchase prediction

**Task.** The same binary classification as `01_baselines.ipynb`: predict whether a customer buys
at least once between 1 and 30 October 2024 (`target` = 1), using features dated on or before
30 September 2024.

**This variant.** A multi-layer perceptron (MLP) where each text column (tier, country, region,
region tier, channel, language) gets a learned **embedding**: a short vector of numbers per
category that the network learns, instead of one-hot columns.

**Question it answers.** *Does learning about categories beat one-hot encoding?* To answer it
fairly, the notebook trains two networks with the same code, the same numeric inputs, the same
settings search and the same split. The only difference is how the text columns go in:

| Model | Text columns enter as |
|---|---|
| MLP + embeddings | one learned vector per category |
| MLP + one-hot | one-hot columns (one 0/1 column per category) |

Both are then compared with the four baselines in `results/baseline_results.csv`.

**Contents**

| Section | What it does |
|---|---|
| 0 | Setup |
| 1 | Load the data and the shared split |
| 2 | EDA of the text columns (train split only) |
| 3 | Preprocessing, fitted on the train split only |
| 4 | The network |
| 5 | A first training run, with learning curves |
| 6 | Tuning: random search, with a tuning log |
| 7 | Seed check: is the difference real or luck? |
| 8 | Final evaluation: the test split is used once, here |
| 9 | What did the embeddings learn? |
| 10 | Save the model |
| 11 | Findings |

**How to run.** Run `01_baselines.ipynb` once first, so `splits.csv` exists (and, for the
precision-recall plot, the saved baseline models). Then select the `.venv` kernel and *Run All*.
Seed 42 everywhere. Section 6 trains 40 networks on the CPU and is the slow part; lower `N_TRIALS`
for a quick run.

## 0. Setup

The network, its preprocessing and the training loop live in `src/embedding_mlp.py`, with unit
tests in `tests/test_embedding_mlp.py`. The split, the baselines' preprocessor and the metrics come
from `src/data_prep.py`, so the numbers here are directly comparable with the baselines.

In [1]:
import sys
import time
from pathlib import Path

# Find the project root (the folder that contains src/data_prep.py), so the notebook works
# from the root or from notebooks/.
ROOT = Path.cwd()
while not (ROOT / "src" / "data_prep.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.preprocessing import OneHotEncoder

# Shared team code: same split, same metrics as the baselines
from data_prep import (
    CATEGORICAL_COLS, MODELS_DIR, RESULTS_DIR, SEED,
    best_f1_threshold, evaluate, get_splits, load_data, precision_at_top, split_xy,
)
# This variant's code
from embedding_mlp import (
    EmbeddingMLP, TabularEncoder, TrainConfig, embedding_dims, load_model, predict_proba,
    sample_config, save_model, set_seed, train_model,
)

set_seed(SEED)                            # fixed seed everywhere (SEED = 42)
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)

# One fixed colour per model, the same as in 01_baselines for the shared ones.
COLOURS = {
    "MLP + embeddings": "#7a4fc9", "MLP + one-hot": "#c94f8a",
    "Recency rule": "#eb6834", "Logistic regression": "#1baf7a", "LightGBM": "#2a78d6",
}
GRID = "#e4e3df"

print("Project root:", ROOT.name, "| torch", torch.__version__, "| seed", SEED)

Project root: AI-Innovation-with-Deep-Learning | torch 2.15.0+cpu | seed 42


## 1. Load the data and the shared split

`get_splits()` loads `splits.csv`, the 70 / 15 / 15 split created by `01_baselines.ipynb`, so this
notebook trains and evaluates on exactly the same customers as the baselines.

In [2]:
df = load_data()
train, val, test = get_splits(df)

X_train, y_train = split_xy(train)        # customer_id is dropped inside split_xy
X_val, y_val = split_xy(val)
X_test, y_test = split_xy(test)

print(f"Train {len(X_train):,} | validation {len(X_val):,} | test {len(X_test):,} customers")
print(f"Features: {X_train.shape[1]} ({len(CATEGORICAL_COLS)} text, "
      f"{X_train.shape[1] - len(CATEGORICAL_COLS)} numeric)")
print(f"Buy rate: train {y_train.mean():.3f}, validation {y_val.mean():.3f}, test {y_test.mean():.3f}")

Train 44,800 | validation 9,600 | test 9,600 customers
Features: 69 (6 text, 63 numeric)
Buy rate: train 0.146, validation 0.146, test 0.146


## 2. EDA of the text columns

This variant only changes how the text columns are used, so this is the part of the EDA it owns:

1. How many categories each column has, and how small the smallest ones are.
2. The buy rate of each category, with a 95% confidence interval, so real differences can be told
   apart from small-sample noise.
3. Whether similar categories behave alike, which is what embeddings can exploit.

**Only the train split is used here.** Looking at validation or test buy rates would leak
information into the choices made later.

In [ ]:
eda = X_train.assign(target=y_train)
base_rate = y_train.mean()


def wilson_interval(buyers, n, z=1.96):
    # 95% confidence interval for a buy rate (Wilson score interval).
    # Narrow for big categories, wide for small ones.
    p = buyers / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def buy_rate_table(col):
    # One row per category: customers, buyers, buy rate and its 95% interval.
    t = eda.groupby(col)["target"].agg(customers="size", buyers="sum")
    t["buy_rate"] = t["buyers"] / t["customers"]
    t["low"], t["high"] = wilson_interval(t["buyers"], t["customers"])
    # "Differs" = the whole interval is above, or below, the overall train buy rate.
    t["differs"] = (t["low"] > base_rate) | (t["high"] < base_rate)
    return t.sort_values("buy_rate", ascending=False)


# --- 2.1 How many categories, and how small the smallest ones are ---------------
unseen = {c: len((set(X_val[c]) | set(X_test[c])) - set(X_train[c])) for c in CATEGORICAL_COLS}
cardinality = pd.DataFrame({
    "categories": X_train[CATEGORICAL_COLS].nunique(),
    "smallest (customers)": [X_train[c].value_counts().min() for c in CATEGORICAL_COLS],
    "largest share": [X_train[c].value_counts(normalize=True).max() for c in CATEGORICAL_COLS],
    "missing": X_train[CATEGORICAL_COLS].isna().sum(),
    "in val/test but not train": pd.Series(unseen),
    "categories that differ": [int(buy_rate_table(c)["differs"].sum()) for c in CATEGORICAL_COLS],
})
display(cardinality.style.format({"smallest (customers)": "{:,}", "largest share": "{:.1%}"}))
print(f"Overall train buy rate: {base_rate:.3f}")

**Reading the table.** *Categories that differ* counts the categories whose buy rate is clearly
above or below the overall rate (the 95% interval does not include it). A column where almost no
category differs carries little information, whatever the encoding. A column with many categories
and many that differ (expected: `region`) is where embeddings have the most to offer, because
one-hot gives each of those categories its own unrelated weight.

In [ ]:
# --- 2.2 Buy rate per category, for the columns with few categories -------------
small_cols = [c for c in CATEGORICAL_COLS if X_train[c].nunique() <= 20]
n_cols = 3
n_rows = int(np.ceil(len(small_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 3.6 * n_rows), squeeze=False)

for ax, col in zip(axes.flat, small_cols):
    t = buy_rate_table(col)
    y_pos = np.arange(len(t))[::-1]
    ax.barh(y_pos, t["buy_rate"], color="#7a4fc9", height=0.65)
    ax.errorbar(t["buy_rate"], y_pos, xerr=[t["buy_rate"] - t["low"], t["high"] - t["buy_rate"]],
                fmt="none", ecolor="#2b2a28", elinewidth=1, capsize=2)
    ax.axvline(base_rate, color="#52514e", linestyle="--", linewidth=1.2)
    ax.set_yticks(y_pos, [f"{lvl} ({n:,})" for lvl, n in zip(t.index, t["customers"])])
    ax.set_title(col, loc="left", fontweight="bold")
    ax.grid(axis="x", color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.flat[len(small_cols):]:
    ax.axis("off")

fig.suptitle("Buy rate by category (train split), with 95% intervals. "
             "Dashed line = overall rate. Customers in brackets.", x=0.01, ha="left")
fig.tight_layout()
plt.show()

In [ ]:
# --- 2.3 Region: many categories, so check which differences are real ------------
regions = buy_rate_table("region")
print(f"{len(regions)} regions; {int(regions['differs'].sum())} have a buy rate clearly different "
      f"from the overall {base_rate:.3f}. Region sizes: {regions['customers'].min():,} to "
      f"{regions['customers'].max():,} customers.")

fig, ax = plt.subplots(figsize=(13, 4.5))
x = np.arange(len(regions))
colour = np.where(regions["differs"], "#7a4fc9", "#b9b7b0")
ax.vlines(x, regions["low"], regions["high"], color=colour, linewidth=1.5)
ax.scatter(x, regions["buy_rate"], color=colour, s=14, zorder=3)
ax.axhline(base_rate, color="#52514e", linestyle="--", linewidth=1.2)
ax.set_xticks([])
ax.set_xlabel("Region, sorted by buy rate (purple = interval excludes the overall rate)")
ax.set_ylabel("Buy rate, with 95% interval")
ax.set_title("Buy rate by region (train split)", loc="left", fontweight="bold")
ax.grid(axis="y", color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

display(pd.concat([regions.head(5), regions.tail(5)])
        .style.format({"buy_rate": "{:.3f}", "low": "{:.3f}", "high": "{:.3f}"}))

In [ ]:
# --- 2.4 Do similar categories behave alike? ---------------------------------------
# Embeddings can only help if categories share structure, for example regions in the same
# country, or of the same region tier, buying at similar rates.
countries_per_region = eda.groupby("region")["country_code"].nunique()
print("Every region belongs to exactly one country:", bool((countries_per_region == 1).all()))

region_info = (eda.groupby("region")
               .agg(country=("country_code", "first"), region_tier=("region_tier", "first"),
                    customers=("target", "size"), buy_rate=("target", "mean")))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, group in zip(axes, ["country", "region_tier"]):
    levels = sorted(region_info[group].unique())
    for i, level in enumerate(levels):
        part = region_info[region_info[group] == level]
        jitter = np.random.default_rng(SEED).uniform(-0.15, 0.15, len(part))
        ax.scatter(np.full(len(part), i) + jitter, part["buy_rate"],
                   s=part["customers"] / part["customers"].max() * 80 + 8,
                   color="#7a4fc9", alpha=0.6, edgecolor="none")
    ax.axhline(base_rate, color="#52514e", linestyle="--", linewidth=1.2)
    ax.set_xticks(range(len(levels)), levels)
    ax.set_xlabel(group)
    ax.set_title(f"Region buy rates grouped by {group}", loc="left", fontweight="bold")
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Region buy rate (dot size = customers)")
fig.tight_layout()
plt.show()

# How much of the spread between regions is explained by the group they are in?
# Share of variance between groups (customer-weighted). 1 = the group explains it all.
for group in ["country", "region_tier"]:
    w = region_info["customers"]
    overall = np.average(region_info["buy_rate"], weights=w)
    group_mean = region_info.groupby(group)["buy_rate"].transform(
        lambda s: np.average(s, weights=w[s.index]))
    between = np.average((group_mean - overall) ** 2, weights=w)
    total = np.average((region_info["buy_rate"] - overall) ** 2, weights=w)
    print(f"{group:<12}: explains {between / total:.0%} of the spread in region buy rates")

**EDA findings (fill in after running).**

- *Categories and sizes:* which columns have many categories, and how small is the smallest one?
  Any category in validation or test that is not in train? (Those get the "unknown" embedding.)
- *Which columns matter:* which columns have categories that clearly differ from the overall rate,
  and by how much?
- *Region:* how many of the regions really differ, and how many are within noise?
- *Similar categories:* how much of the region spread do country and region tier explain? If it is
  a lot, regions in the same group should end up with similar embeddings (checked in section 9).

## 3. Preprocessing

`TabularEncoder` (in `src/embedding_mlp.py`) turns the feature table into the two inputs the network
needs:

| Columns | Steps | Output |
|---|---|---|
| Numeric | `log1p` on long-tailed columns → `SimpleImputer(strategy="median")` → `StandardScaler` | float matrix |
| Text | each category → an ID, 1 to *n*, from the categories seen in **train** | integer matrix |

**Why the log step (different from the baselines).** Columns such as `spend_total` or
`credit_limit` have a long right tail: after plain scaling a few very large customers sit far out
while everyone else is squeezed together near 0. Trees (LightGBM) are not affected, because they
only use the order of the values, but a neural network learns badly from it. `log1p(x) = log(1 + x)`
pulls the tail in and keeps the order. A column is logged if, in **train**, it is never negative,
is not a 0/1 flag, and has skewness above 2. The table below lists them.

ID 0 is kept for a category never seen in train. Its embedding is fixed at zeros, which is what a
one-hot encoder with `handle_unknown="ignore"` does with an all-zero row.

**The one-hot comparison network** gets exactly the same numeric matrix, plus one-hot columns for
the text columns instead of IDs. So the two networks differ *only* in how categories go in, which is
the question this notebook asks.

Everything is fitted on the **train split only** and then applied unchanged to validation and test.

**Embedding sizes.** A column with *n* categories gets about 1.6 × *n*^0.56 numbers per category, a
rule of thumb from the fast.ai library, capped at `max_emb_dim`. Both the multiplier (`emb_scale`)
and the cap are tuned in section 6.

In [ ]:
# Embedding model inputs: fitted on TRAIN only.
encoder = TabularEncoder(CATEGORICAL_COLS).fit(X_train)
emb_train, emb_val, emb_test = (encoder.transform(X) for X in (X_train, X_val, X_test))

display(pd.DataFrame({
    "categories (train)": encoder.cardinalities,
    "default embedding size": embedding_dims(encoder.cardinalities),
    "unknown IDs in val": (emb_val[1] == 0).sum(axis=0),
    "unknown IDs in test": (emb_test[1] == 0).sum(axis=0),
}, index=CATEGORICAL_COLS))

# Which numeric columns got the log, and how long their tail was in train.
log_table = pd.DataFrame({
    "skew before": X_train[encoder.log_cols_].skew(),
    "skew after log1p": np.log1p(X_train[encoder.log_cols_].clip(lower=0)).skew(),
})
print(f"{len(encoder.log_cols_)} of {len(encoder.numeric_cols_)} numeric columns are logged:")
display(log_table.sort_values("skew before", ascending=False).style.format("{:.1f}"))

# One-hot model inputs: the SAME numeric matrix, plus one-hot columns for the text columns.
# Fitted on TRAIN only; a category unseen in train becomes an all-zero row.
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
onehot.fit(X_train[CATEGORICAL_COLS])


def to_onehot_inputs(X):
    # Same (numeric, categorical) pair as TabularEncoder, with the one-hot columns joined to the
    # numeric part and no category IDs, so the same network code builds a plain MLP.
    x_num, _ = encoder.transform(X)
    m = np.hstack([x_num, onehot.transform(X[CATEGORICAL_COLS])]).astype(np.float32)
    return m, np.zeros((len(X), 0), dtype=np.int64)


oh_train, oh_val, oh_test = (to_onehot_inputs(X) for X in (X_train, X_val, X_test))

print(f"Embedding model input: {emb_train[0].shape[1]} numeric columns + "
      f"{emb_train[1].shape[1]} category IDs")
print(f"One-hot model input  : {oh_train[0].shape[1]} columns "
      f"({emb_train[0].shape[1]} numeric + {oh_train[0].shape[1] - emb_train[0].shape[1]} one-hot)")
print("Missing values left:", int(np.isnan(emb_val[0]).sum()), "and", int(np.isnan(oh_val[0]).sum()))

## 4. The network

```
numeric columns ── scaled ─────────────────────────────┐
country_code ── ID ── Embedding(categories + 1, d) ────┤
...one embedding table per text column ...             ├─ concat ─ [Dense → BatchNorm → ReLU → Dropout] × k ─ Dense(1) ─ sigmoid
region ──────── ID ── Embedding(categories + 1, d) ────┘
```

- **Embedding tables** start random and are learned with the rest of the network.
- **Dropout on the embeddings** (`emb_dropout`) and after each Dense layer (`dropout`) reduces
  overfitting.
- **Output:** one number per customer. The sigmoid turns it into a probability between 0 and 1.
  During training the sigmoid is inside the loss (`BCEWithLogitsLoss`), which is the numerically safe
  way to do the same thing.
- **Loss:** binary cross-entropy. `pos_weight` makes each missed buyer cost more than a wrongly
  flagged non-buyer, to cope with only 14.6% buyers (the same idea as LightGBM's `scale_pos_weight`).

In [ ]:
default_config = TrainConfig()
example = EmbeddingMLP(
    n_numeric=emb_train[0].shape[1],
    cardinalities=encoder.cardinalities,
    emb_dims=embedding_dims(encoder.cardinalities, default_config.emb_scale,
                            default_config.max_emb_dim),
    hidden=default_config.hidden,
    dropout=default_config.dropout,
    emb_dropout=default_config.emb_dropout,
)
print(example)

n_emb = sum(p.numel() for p in example.embeddings.parameters())
n_all = sum(p.numel() for p in example.parameters())
print(f"\nParameters: {n_all:,} in total, {n_emb:,} of them in the embedding tables.")

## 5. A first training run

One network with the default settings, to check that training behaves before tuning anything.

**Early stopping.** After every epoch (one pass over the train split) the network is scored on
**validation** PR-AUC. Training stops when that score has not improved for `patience` (10) epochs,
and the weights from the best epoch are kept. Validation is used only to decide when to stop, never
to update the weights.

**Learning-rate schedule.** If validation PR-AUC has not improved for `lr_patience` (3) epochs, the
learning rate is halved (`ReduceLROnPlateau`). Smaller steps often find a slightly better point
before early stopping gives up. The rate used in each epoch is in the `lr` column of the history.

**Divergence guard.** If the loss becomes infinite or NaN (a learning rate that is far too high),
training stops and keeps its best epoch so far, so one bad trial cannot crash the search in section 6.

In [ ]:
def plot_history(history, title):
    # Loss curves (left) and validation PR-AUC (right); the dotted line marks the kept epoch.
    best = history.attrs["best_epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].plot(history["epoch"], history["train_loss"], color="#2a78d6", label="train")
    axes[0].plot(history["epoch"], history["val_loss"], color="#eb6834", label="validation")
    axes[0].set_ylabel("Loss (weighted cross-entropy)")
    axes[0].legend(frameon=False)
    axes[1].plot(history["epoch"], history["val_pr_auc"], color="#7a4fc9")
    axes[1].set_ylabel("Validation PR-AUC")
    for ax in axes:
        ax.axvline(best, color="#52514e", linestyle=":", linewidth=1.2)
        ax.set_xlabel("Epoch")
        ax.grid(color=GRID)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{title}: best epoch {best}, validation PR-AUC "
                 f"{history.attrs['best_val_pr_auc']:.4f}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    plt.show()


started = time.perf_counter()
first_model, first_history = train_model(
    emb_train, y_train, emb_val, y_val, encoder.cardinalities, default_config, seed=SEED)
print(f"Trained {len(first_history)} epochs in {time.perf_counter() - started:.0f} s")
plot_history(first_history, "MLP + embeddings, default settings")

**Reading the curves.** If validation loss turns upwards while train loss keeps falling, the network
is overfitting; early stopping keeps the epoch before that. If both curves are still falling at the
end, it needs more epochs or a higher learning rate. Note what you see here: it guides the search
space in section 6.

## 6. Tuning: random search

Random search tries `N_TRIALS` combinations of settings drawn from the lists below. It covers a
large space better than a grid of the same size, because every trial tries a new value of every
setting.

**Fair comparison.** Exactly the same `N_TRIALS` settings are tried for both networks, with the same
seed. The embedding settings (`emb_*`, `max_emb_dim`) do not apply to the one-hot network and are
left blank in its rows. Every trial is judged on **validation PR-AUC**, and every trial is saved to
`results/mlp_embeddings_tuning_log.csv`, which is the evidence of optimisation.

`pos_weight` options: 1 (no weighting), the square root of the class ratio, and the full class ratio
(non-buyers ÷ buyers in train, the "balanced" value LightGBM used).

In [ ]:
N_TRIALS = 20                     # per network; lower it for a quick run

neg_pos = float((y_train == 0).sum() / (y_train == 1).sum())
SEARCH_SPACE = {
    "hidden": [(64,), (128, 64), (256, 128), (256, 128, 64)],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.5],
    "emb_dropout": [0.0, 0.1, 0.2, 0.3],
    "emb_scale": [0.5, 1.0, 1.5, 2.0],
    "max_emb_dim": [8, 16, 32],
    "lr": [3e-4, 1e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3],
    "batch_size": [256, 512, 1024],
    "pos_weight": [1.0, round(np.sqrt(neg_pos), 2), round(neg_pos, 2)],
}
rng = np.random.default_rng(SEED)
configs = [sample_config(rng, SEARCH_SPACE) for _ in range(N_TRIALS)]
EMB_ONLY = ["emb_dropout", "emb_scale", "max_emb_dim"]


def run_search(name, tr, va, cardinalities):
    # Train one network per config; return the log rows and the trained models.
    rows, fitted = [], {}
    for trial, config in enumerate(configs):
        started = time.perf_counter()
        model, history = train_model(tr, y_train, va, y_val, cardinalities, config, seed=SEED)
        row = {"model": name, "trial": trial, **config.as_dict(),
               "best_epoch": history.attrs["best_epoch"], "epochs_run": len(history),
               "val_pr_auc": history.attrs["best_val_pr_auc"],
               "seconds": round(time.perf_counter() - started, 1)}
        if not cardinalities:
            row.update(dict.fromkeys(EMB_ONLY, np.nan))
        rows.append(row)
        fitted[trial] = model
        print(f"{name} trial {trial + 1:2d}/{N_TRIALS}: val PR-AUC {row['val_pr_auc']:.4f} "
              f"({row['epochs_run']} epochs, {row['seconds']:.0f} s)")
    return rows, fitted


emb_rows, emb_models = run_search("MLP + embeddings", emb_train, emb_val, encoder.cardinalities)
oh_rows, oh_models = run_search("MLP + one-hot", oh_train, oh_val, [])

tuning_log = pd.DataFrame(emb_rows + oh_rows)
tuning_log.to_csv(RESULTS_DIR / "mlp_embeddings_tuning_log.csv", index=False)
print("\nSaved:", (RESULTS_DIR / "mlp_embeddings_tuning_log.csv").relative_to(ROOT).as_posix())

In [ ]:
# Best 5 trials of each network.
show = ["trial", "hidden", "dropout", "emb_dropout", "emb_scale", "max_emb_dim", "lr",
        "weight_decay", "batch_size", "pos_weight", "best_epoch", "val_pr_auc"]
for name, part in tuning_log.groupby("model", sort=False):
    print(name)
    display(part.sort_values("val_pr_auc", ascending=False)[show].head(5)
            .style.format({"val_pr_auc": "{:.4f}", "lr": "{:g}", "weight_decay": "{:g}"},
                          na_rep="-").hide(axis="index"))

# Same settings, two encodings: one dot per trial. Above the diagonal = embeddings did better.
paired = tuning_log.pivot(index="trial", columns="model", values="val_pr_auc")
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(paired["MLP + one-hot"], paired["MLP + embeddings"], color="#7a4fc9", s=30)
lims = [paired.min().min() - 0.005, paired.max().max() + 0.005]
ax.plot(lims, lims, color="#52514e", linestyle="--", linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("MLP + one-hot: validation PR-AUC")
ax.set_ylabel("MLP + embeddings: validation PR-AUC")
ax.set_title("Same settings, two encodings", loc="left", fontweight="bold")
ax.grid(color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

wins = int((paired["MLP + embeddings"] > paired["MLP + one-hot"]).sum())
print(f"Embeddings scored higher in {wins} of {len(paired)} trials; median difference "
      f"{(paired['MLP + embeddings'] - paired['MLP + one-hot']).median():+.4f} PR-AUC.")

**Tuning findings (fill in after running).** Which settings did the best trials share (depth, dropout,
`pos_weight`, embedding size)? Did the best trials stop early, or run near `max_epochs` (a sign the
search space should allow more)? How many trials did embeddings win?

## 7. Seed check: is the difference real or luck?

A network's score changes with its random seed (starting weights, batch order). If the two networks
differ by less than that, the difference is luck. So the best settings of each network are retrained
with three seeds and the spread is compared with the gap.

The **final model** of each network is still its seed-42 run from the search: seeds are not
cherry-picked.

In [ ]:
SEEDS = [SEED, SEED + 1, SEED + 2]
inputs = {"MLP + embeddings": (emb_train, emb_val, encoder.cardinalities, emb_models),
          "MLP + one-hot": (oh_train, oh_val, [], oh_models)}

best_trial, final_models, seed_rows = {}, {}, []
for name, (tr, va, cards, fitted) in inputs.items():
    part = tuning_log[tuning_log["model"] == name]
    best_trial[name] = int(part.loc[part["val_pr_auc"].idxmax(), "trial"])
    final_models[name] = fitted[best_trial[name]]          # seed 42, from the search
    for seed in SEEDS:
        if seed == SEED:
            score = part.loc[part["trial"] == best_trial[name], "val_pr_auc"].item()
        else:
            _, history = train_model(tr, y_train, va, y_val, cards, configs[best_trial[name]],
                                     seed=seed)
            score = history.attrs["best_val_pr_auc"]
        seed_rows.append({"model": name, "seed": seed, "val_pr_auc": score})

seeds = pd.DataFrame(seed_rows)
seed_summary = seeds.groupby("model", sort=False)["val_pr_auc"].agg(["mean", "std", "min", "max"])
display(seed_summary.style.format("{:.4f}"))

gap = seed_summary.loc["MLP + embeddings", "mean"] - seed_summary.loc["MLP + one-hot", "mean"]
spread = seed_summary["std"].max()
print(f"Gap (embeddings - one-hot): {gap:+.4f}. Largest seed-to-seed std: {spread:.4f}.")
print("A gap smaller than about 2 x the std should be read as a tie.")

## 8. Final evaluation: validation and test

Settings and models are frozen. Each network's threshold is chosen on **validation** (best F1), then
the **test split is used once**, with the same `evaluate()` as the baselines. The results are saved to
`results/mlp_embeddings_results.csv` in the same format as `results/baseline_results.csv`.

In [ ]:
encoded = {"MLP + embeddings": (emb_val, emb_test), "MLP + one-hot": (oh_val, oh_test)}
val_results, test_results, test_scores = {}, {}, {}
for name, model in final_models.items():
    va, te = encoded[name]
    val_scores = predict_proba(model, va)
    threshold = best_f1_threshold(y_val, val_scores)           # chosen on validation
    val_results[name] = evaluate(y_val, val_scores, threshold)
    test_scores[name] = predict_proba(model, te)               # the one use of test
    test_results[name] = evaluate(y_test, test_scores[name], threshold)


def results_table(results, split_name):
    table = pd.DataFrame(results).T
    table.insert(0, "split", split_name)
    table["accuracy"] = np.nan                                 # as in baseline_results.csv
    table.index.name = "model"
    return table


nn_results = pd.concat([results_table(val_results, "val"),
                        results_table(test_results, "test")]).reset_index()
nn_results.to_csv(RESULTS_DIR / "mlp_embeddings_results.csv", index=False)
print("Saved:", (RESULTS_DIR / "mlp_embeddings_results.csv").relative_to(ROOT).as_posix())

# --- Compare with the baselines -------------------------------------------------
baselines = pd.read_csv(RESULTS_DIR / "baseline_results.csv")
everything = pd.concat([baselines, nn_results], ignore_index=True)

metric_cols = ["pr_auc", "roc_auc", "precision", "recall", "f1", "precision_top10", "lift_top10"]
fmt = {c: "{:.3f}" for c in metric_cols} | {"lift_top10": "{:.2f}x"}
for split_name, label in [("val", "VALIDATION"), ("test", "TEST: the reported numbers")]:
    print(label)
    table = everything[everything["split"] == split_name].set_index("model")[metric_cols]
    display(table.astype(float).style.format(fmt))

# Gap to the strongest baseline on test PR-AUC.
test_table = everything[everything["split"] == "test"].set_index("model")
best_baseline = test_table.loc[baselines["model"].unique(), "pr_auc"].idxmax()
for name in final_models:
    diff = test_table.loc[name, "pr_auc"] - test_table.loc[best_baseline, "pr_auc"]
    verdict = "within noise (under 0.01)" if abs(diff) < 0.01 else ("better" if diff > 0 else "worse")
    print(f"{name}: test PR-AUC {diff:+.4f} vs {best_baseline} -> {verdict}")

In [ ]:
# --- Precision-recall curves on test, with the baselines when their models are saved ---
curves = dict(test_scores)
try:
    logreg = joblib.load(MODELS_DIR / "logreg_baseline.joblib")
    curves["Logistic regression"] = logreg.predict_proba(X_test)[:, 1]
    lgbm = joblib.load(MODELS_DIR / "lightgbm_baseline.joblib")
    levels = {c: sorted(X_train[c].unique()) for c in CATEGORICAL_COLS}   # as in 01_baselines
    X_test_lgbm = X_test.assign(**{c: pd.Categorical(X_test[c], categories=levels[c])
                                   for c in CATEGORICAL_COLS})
    curves["LightGBM"] = lgbm.predict_proba(X_test_lgbm)[:, 1]
except (FileNotFoundError, ImportError) as e:
    print("Baseline models not available, plotting the networks only:", e)

fig, ax = plt.subplots(figsize=(8, 5.5))
for name, scores in curves.items():
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ax.plot(recall, precision, color=COLOURS[name], linewidth=2,
            label=f"{name} (PR-AUC {average_precision_score(y_test, scores):.3f})")
ax.axhline(y_test.mean(), color="#52514e", linewidth=1.5, linestyle="--",
           label=f"Always 0 / random (PR-AUC {y_test.mean():.3f})")
ax.set_xlabel("Recall (share of buyers found)")
ax.set_ylabel("Precision (share of flagged customers who bought)")
ax.set_title("Precision-recall curves on the test split", loc="left", fontweight="bold")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "mlp_embeddings_pr_curves.png", dpi=150)
plt.show()

# --- The business view: buyers among each model's top 1,000 test customers --------
TOP_N = 1000
for name, scores in curves.items():
    found = precision_at_top(y_test, scores, TOP_N / len(y_test)) * TOP_N
    print(f"{name:<20}: {found:,.0f} of the top {TOP_N:,} bought "
          f"(random: ~{TOP_N * y_test.mean():,.0f})")

## 9. What did the embeddings learn?

If the embeddings captured something real, categories with similar buy rates should have similar
vectors. Two checks, both on the final embedding model and the **train** buy rates:

1. **Per column:** reduce each column's vectors to their first principal component and check how
   well it orders the categories by buy rate (Spearman correlation, sign ignored: the direction of a
   component is arbitrary).
2. **Region map:** the region vectors reduced to two dimensions, coloured by buy rate. Regions with
   similar colours close together means the network grouped regions that behave alike.

The network can also use embeddings for things other than the overall buy rate (for example, how a
region interacts with recency), so a weak correlation is not by itself a failure.

In [ ]:
emb_model = final_models["MLP + embeddings"]


def embedding_frame(col):
    # One row per category seen in train: its learned vector, customers and train buy rate.
    i = CATEGORICAL_COLS.index(col)
    vectors = emb_model.embeddings[i].weight.detach().numpy()[1:]     # row 0 = unknown
    levels = list(encoder.vocab_[col])                                # IDs 1..n in order
    rates = eda.groupby(col)["target"].agg(customers="size", buy_rate="mean").reindex(levels)
    return vectors, rates


def principal_components(vectors, k=2):
    centred = vectors - vectors.mean(axis=0)
    u, s, _ = np.linalg.svd(centred, full_matrices=False)
    k = min(k, u.shape[1])
    return u[:, :k] * s[:k], (s**2 / (s**2).sum())[:k]


rows = []
for col in CATEGORICAL_COLS:
    vectors, rates = embedding_frame(col)
    if len(rates) < 3:
        continue
    pcs, explained = principal_components(vectors, k=1)
    corr = pd.Series(pcs[:, 0]).corr(pd.Series(rates["buy_rate"].to_numpy()), method="spearman")
    rows.append({"column": col, "categories": len(rates), "embedding size": vectors.shape[1],
                 "variance in 1st component": explained[0], "|Spearman| with buy rate": abs(corr)})
display(pd.DataFrame(rows).set_index("column").style.format(
    {"variance in 1st component": "{:.0%}", "|Spearman| with buy rate": "{:.2f}"}))

In [ ]:
vectors, rates = embedding_frame("region")
pcs, explained = principal_components(vectors, k=2)

fig, ax = plt.subplots(figsize=(8, 6))
points = ax.scatter(pcs[:, 0], pcs[:, 1], c=rates["buy_rate"], cmap="viridis",
                    s=rates["customers"] / rates["customers"].max() * 120 + 10,
                    edgecolor="white", linewidth=0.5)
fig.colorbar(points, ax=ax, label="Train buy rate")
ax.set_xlabel(f"Component 1 ({explained[0]:.0%} of the variance)")
ax.set_ylabel(f"Component 2 ({explained[1]:.0%} of the variance)")
ax.set_title("Learned region embeddings, reduced to 2D (dot size = customers)",
             loc="left", fontweight="bold")
ax.grid(color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "mlp_embeddings_region_map.png", dpi=150)
plt.show()

# Nearest neighbours: are the most similar regions also similar in buy rate (and country)?
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True).clip(1e-12)
similarity = unit @ unit.T
names = rates.index.to_list()
for region in [rates["buy_rate"].idxmax(), rates["buy_rate"].idxmin()]:
    i = names.index(region)
    nearest = [j for j in np.argsort(-similarity[i]) if j != i][:3]
    print(f"{region} (buy rate {rates['buy_rate'].iloc[i]:.3f}, "
          f"{region_info.loc[region, 'country']}) is closest to: " +
          ", ".join(f"{names[j]} ({rates['buy_rate'].iloc[j]:.3f}, "
                    f"{region_info.loc[names[j], 'country']})" for j in nearest))

## 10. Save the model

| File | Contents |
|---|---|
| `models/mlp_embeddings.pt` | The final embedding network: architecture and weights |
| `models/mlp_embeddings_encoder.joblib` | The `TabularEncoder` fitted on train: medians, scaling and category IDs |

`models/` is git-ignored, so teammates create these files by running this notebook.

In [ ]:
save_model(emb_model, MODELS_DIR / "mlp_embeddings.pt")
joblib.dump(encoder, MODELS_DIR / "mlp_embeddings_encoder.joblib")

# Reload both and confirm they give exactly the same validation scores.
reloaded = load_model(MODELS_DIR / "mlp_embeddings.pt")
encoder_reloaded = joblib.load(MODELS_DIR / "mlp_embeddings_encoder.joblib")
assert np.allclose(predict_proba(reloaded, encoder_reloaded.transform(X_val)),
                   predict_proba(emb_model, emb_val))
print("Saved and verified: models/mlp_embeddings.pt, models/mlp_embeddings_encoder.joblib")

### How a teammate reuses this

```python
import sys
sys.path.insert(0, "src")                         # "../src" from inside notebooks/
import joblib
from embedding_mlp import load_model, predict_proba

model = load_model("models/mlp_embeddings.pt")
encoder = joblib.load("models/mlp_embeddings_encoder.joblib")
scores = predict_proba(model, encoder.transform(X))  # X = feature table without customer_id/target
```

## 11. Findings

All model numbers are from the test split unless stated. Run of 17 November 2026: `N_TRIALS = 20` per
network, seed 42.

| Model | PR-AUC | ROC-AUC | F1 | Precision in top 10% | Lift |
|---|---|---|---|---|---|
| Logistic regression (best baseline) | 0.577 | 0.873 | 0.543 | 0.626 | 4.3x |
| LightGBM | 0.576 | 0.874 | 0.543 | 0.605 | 4.2x |
| MLP + one-hot | 0.583 | 0.875 | 0.547 | 0.632 | 4.3x |
| MLP + embeddings | 0.575 | 0.875 | 0.543 | 0.620 | 4.3x |

**1. Does learning about categories beat one-hot encoding? No: it is a tie.** In the seed check
(section 7) the best embedding settings average 0.5638 validation PR-AUC over three seeds and the best
one-hot settings 0.5647. The gap, -0.0009, is far smaller than the seed-to-seed spread (std up to
0.0034). In the search, embeddings scored higher in only 6 of the 20 paired trials (median difference
-0.0013). On test, one-hot is 0.008 ahead, but that is one seed-42 model each and still under the
0.01 bar. The EDA explains why: only `region` (14 of 100 regions) and `customer_tier` have categories
whose buy rate clearly differs from the overall rate, and the numeric columns probably carry most of
the signal, so how the text columns go in changes little.

**2. Does either network beat the baselines? No.** Against logistic regression on test PR-AUC, MLP +
one-hot is +0.006 and MLP + embeddings is -0.002, both within noise (under 0.01). ROC-AUC (0.875),
F1 (0.54 to 0.55) and lift (about 4.3x) are level with the baselines too. In the business view, 622
(one-hot) and 607 (embeddings) of the top 1,000 test customers bought, against about 146 at random.
All four models seem to reach the same ceiling on these features, so a better score now needs better
features rather than a different model.

**3. What the embeddings learned: some ordering, little grouping.** The first principal component
orders the categories by buy rate well for `preferred_language` (|Spearman| 1.0) and
`preferred_channel` (0.8), but these have only 4 categories each, and the EDA found none of them
clearly differs from the overall rate, so these correlations are weak evidence. `customer_tier`, whose
4 categories all differ, shows no ordering (0.0), probably because spend and order columns already
carry the tier's information. `region` is weak (0.2). On the region map, the highest-rate region
(region_23, 0.223) sits next to other high-rate regions (0.186 to 0.214) from both the US and UK, but
the lowest (region_89, 0.077, only 130 customers) sits next to average regions. This matches the EDA
in 2.4: country explains only 3% and region tier 1% of the spread between regions, so there is little
shared group structure for the embeddings to exploit.

**4. What the tuning showed: the settings barely matter.** Across 20 trials, validation PR-AUC ranged
only from 0.550 to 0.568 (embeddings) and 0.557 to 0.564 (one-hot). The highest learning rate
(0.003) was used by all five of the best one-hot trials and by the top three embedding trials, and the
middle `pos_weight` (2.42) was the most common among the best trials, although no single value won.
Depth made no consistent difference: the best embedding trial was a single 64-unit layer with heavy
dropout (0.5) and small embeddings (`emb_scale` 2.0, capped at 8), while the best one-hot trial had
three layers. Every trial stopped early, well before `max_epochs` (100): the longest ran 62 epochs and
the best epochs were 2 to 52, so the epoch limit did not hold the search back. The tuning log does not
record each trial's learning-rate history, so whether the rate was halved before the best epoch was
not checked here.

**5. Limits.** The best embedding trial's seed-42 score (0.5675) is the highest of its three seeds,
so the final embedding model is a favourable draw; the three-seed mean (0.5638) is the fairer figure.
`pos_weight` above 1 inflates the probabilities, so they are ranking scores, not calibrated
probabilities (calibration is planned for Progress Review 2). The networks use a `log1p` step on
long-tailed columns that the baselines do not, so a network/baseline gap mixes the model and that
step; the two networks share it, so their comparison does not. The saved baseline models were not in
`models/` for this run, so the precision-recall plot shows the two networks only; run
`01_baselines.ipynb` first to add the baselines. The `loyalty_score` leakage check from the baselines
notebook applies here too.